# Sentiment Analysis

### Setup Spark

In [2]:
import time
import pandas as pd
pd.DataFrame.iteritems = pd.DataFrame.items
import random
import json
import pyspark.pandas as ps
from pyspark.sql.functions import regexp_replace, udf, length, split, array_except, col, \
                                  lit, explode_outer, explode, regexp_replace, collect_list, \
                                  concat_ws, broadcast
from pyspark.sql.types import StructType, StructField, StringType, ArrayType
from py4j.java_gateway import java_import
from pyspark.sql import functions as F
from pyspark.sql import Row

In [5]:
import findspark
findspark.init('/home/hrn4ch/spark-3.3.1-bin-hadoop3')
findspark.find()

'/home/hrn4ch/spark-3.3.1-bin-hadoop3'

In [6]:
# import dependencies, set env variables

import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64/jre"
os.environ["SPARK_HOME"] = "/home/hrn4ch/spark-3.3.1-bin-hadoop3"
os.environ['PYSPARK_SUBMIT_ARGS'] = "--master local[3] pyspark-shell"

In [7]:
from pyspark.sql import SparkSession

# The entry point into all functionality in Spark is the SparkSession class.
spark = (SparkSession
	.builder
	.appName("FinalProj")
	.master("spark://10.150.0.2:7077")
	.config("spark.executor.memory", "1024M")
	.getOrCreate())

# You can read the data from a file into DataFrames
df_reviews = spark.read.json("hdfs://10.150.0.2:9000/yelp_academic_dataset_review.json")
df_business = spark.read.json("hdfs://10.150.0.2:9000/yelp_academic_dataset_business.json")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


24/04/23 02:52:47 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


24/04/23 02:53:28 WARN package: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


In [8]:
df_reviews = df_reviews.withColumnRenamed("stars", "review_stars")
df_reviews.show(5)

+--------------------+----+-------------------+-----+--------------------+------------+--------------------+------+--------------------+
|         business_id|cool|               date|funny|           review_id|review_stars|                text|useful|             user_id|
+--------------------+----+-------------------+-----+--------------------+------------+--------------------+------+--------------------+
|XQfwVwDr-v0ZS3_Cb...|   0|2018-07-07 22:09:11|    0|KU_O5udG6zpxOg-Vc...|         3.0|If you decide to ...|     0|mh_-eMZ6K5RLWhZyI...|
|7ATYjTIgM3jUlt4UM...|   1|2012-01-03 15:28:18|    0|BiTunyQ73aT9WBnpR...|         5.0|I've taken a lot ...|     1|OyoGAe7OKpv6SyGZT...|
|YjUWPpI6HXG530lwP...|   0|2014-02-05 20:30:30|    0|saUsX_uimxRlCVr67...|         3.0|Family diner. Had...|     0|8g_iMtfSiwikVnbP2...|
|kxX2SOes4o-D3ZQBk...|   1|2015-01-04 00:01:03|    0|AqPFMleE6RsU23_au...|         5.0|Wow!  Yummy, diff...|     1|_7bHUi9Uuf5__HHc_...|
|e4Vwtrqf-wpJfwesg...|   1|2017-01-14 20:

In [9]:
df_business = df_business.withColumnRenamed("stars", "business_stars")
df_business.show(5)

+--------------------+--------------------+--------------------+--------------------+-------------+--------------------+-------+----------+------------+--------------------+-----------+------------+--------------+-----+
|             address|          attributes|         business_id|          categories|         city|               hours|is_open|  latitude|   longitude|                name|postal_code|review_count|business_stars|state|
+--------------------+--------------------+--------------------+--------------------+-------------+--------------------+-------+----------+------------+--------------------+-----------+------------+--------------+-----+
|1616 Chapala St, ...|{null, null, null...|Pns2l4eNsfO8kk83d...|Doctors, Traditio...|Santa Barbara|                null|      0|34.4266787|-119.7111968|Abby Rappoport, L...|      93101|           7|           5.0|   CA|
|87 Grasso Plaza S...|{null, null, null...|mpf3x-BjTdTEA3yCZ...|Shipping Centers,...|       Affton|{8:0-18:30, 0:0-0...|

### Merge datasets

In [10]:
merged = df_reviews.join(df_business, how='inner', on='business_id')
merged.show(5)

+--------------------+----+-------------------+-----+--------------------+------------+--------------------+------+--------------------+----------------+--------------------+--------------------+---------------+--------------------+-------+----------+-----------+-----------------+-----------+------------+--------------+-----+
|         business_id|cool|               date|funny|           review_id|review_stars|                text|useful|             user_id|         address|          attributes|          categories|           city|               hours|is_open|  latitude|  longitude|             name|postal_code|review_count|business_stars|state|
+--------------------+----+-------------------+-----+--------------------+------------+--------------------+------+--------------------+----------------+--------------------+--------------------+---------------+--------------------+-------+----------+-----------+-----------------+-----------+------------+--------------+-----+
|---kPU91CF4Lq2-

Number of unique users who have reviewed: 1987929

In [11]:
merged.select('user_id').distinct().count()

1987929

Number of businesses that have been reviewed: 150346

In [12]:
merged.select('business_id').distinct().count()

150346

### Importing NLP Modules

In [13]:
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

In [14]:
nltk.download('punkt')

[nltk_data] Downloading package punkt to /home/hrn4ch/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

Removing exclamation marks to minimize error.

In [15]:
merged = merged.withColumn("text", regexp_replace(merged["text"], "!", ""))
merged.show(5)

+--------------------+----+-------------------+-----+--------------------+------------+--------------------+------+--------------------+----------------+--------------------+--------------------+---------------+--------------------+-------+----------+-----------+-----------------+-----------+------------+--------------+-----+
|         business_id|cool|               date|funny|           review_id|review_stars|                text|useful|             user_id|         address|          attributes|          categories|           city|               hours|is_open|  latitude|  longitude|             name|postal_code|review_count|business_stars|state|
+--------------------+----+-------------------+-----+--------------------+------------+--------------------+------+--------------------+----------------+--------------------+--------------------+---------------+--------------------+-------+----------+-----------+-----------------+-----------+------------+--------------+-----+
|---kPU91CF4Lq2-

In [16]:
# Download stopwords if not already downloaded
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /home/hrn4ch/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [17]:
merged.columns

['business_id',
 'cool',
 'date',
 'funny',
 'review_id',
 'review_stars',
 'text',
 'useful',
 'user_id',
 'address',
 'attributes',
 'categories',
 'city',
 'hours',
 'is_open',
 'latitude',
 'longitude',
 'name',
 'postal_code',
 'review_count',
 'business_stars',
 'state']

In [18]:
stopwords_df = spark.createDataFrame([(word,) for word in stopwords.words('english')], ['word'])
to_word_list = F.udf(lambda words: [row['word'] for row in words], ArrayType(StringType()))
stopwords_list = stopwords_df.select('word').collect()
stopwords_column = [row['word'] for row in stopwords_list]
stopwords_broadcast = spark.sparkContext.broadcast(stopwords_column)
def filter_stopwords(words):
    return [word for word in words if word not in stopwords_broadcast.value]
filtered_text_df = merged.withColumn('words', split(regexp_replace(col('text'), r'[^a-zA-Z0-9\s]', ''), ' ')) \
    .withColumn('filtered_words', F.udf(filter_stopwords)(col('words'))) \
    .withColumn('filtered_text', concat_ws(' ', 'filtered_words'))
result_df = filtered_text_df.select('business_id', 'cool', 'date', 'funny', 'review_id',
                                     'review_stars', 'text', 'useful', 'user_id', 'address', 'attributes',
                                     'categories', 'city', 'hours', 'is_open', 'latitude', 'longitude', 'name',
                                     'postal_code', 'review_count', 'business_stars', 'state', 'filtered_text')
result_df.show(5)

+--------------------+----+-------------------+-----+--------------------+------------+--------------------+------+--------------------+----------------+--------------------+--------------------+---------------+--------------------+-------+----------+-----------+-----------------+-----------+------------+--------------+-----+--------------------+
|         business_id|cool|               date|funny|           review_id|review_stars|                text|useful|             user_id|         address|          attributes|          categories|           city|               hours|is_open|  latitude|  longitude|             name|postal_code|review_count|business_stars|state|       filtered_text|
+--------------------+----+-------------------+-----+--------------------+------------+--------------------+------+--------------------+----------------+--------------------+--------------------+---------------+--------------------+-------+----------+-----------+-----------------+-----------+---------

In [21]:
file_path = 'NRC-Emotion-Lexicon-Wordlevel-v0.92.txt' # path to lexicon file
lexicon_emotion = {} # define an empty directory
with open(file_path, 'r', encoding='utf-8') as file:
    for line in file:
        word, emotion, value = line.strip().split('\t')
        if int(value) == 1:
            if word not in lexicon_emotion:
                lexicon_emotion[word] = []
            lexicon_emotion[word].append(emotion) # emotion lexicon as a python dictionart

# display the emotion dictionary
list(lexicon_emotion.items())[:10]

[('abacus', ['trust']),
 ('abandon', ['fear', 'negative', 'sadness']),
 ('abandoned', ['anger', 'fear', 'negative', 'sadness']),
 ('abandonment', ['anger', 'fear', 'negative', 'sadness', 'surprise']),
 ('abba', ['positive']),
 ('abbot', ['trust']),
 ('abduction', ['fear', 'negative', 'sadness', 'surprise']),
 ('aberrant', ['negative']),
 ('aberration', ['disgust', 'negative']),
 ('abhor', ['anger', 'disgust', 'fear', 'negative'])]

In [22]:
nltk.download('wordnet')

[nltk_data] Downloading package wordnet to /home/hrn4ch/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [23]:
result_df.dtypes

[('business_id', 'string'),
 ('cool', 'bigint'),
 ('date', 'string'),
 ('funny', 'bigint'),
 ('review_id', 'string'),
 ('review_stars', 'double'),
 ('text', 'string'),
 ('useful', 'bigint'),
 ('user_id', 'string'),
 ('address', 'string'),
 ('attributes',
  'struct<AcceptsInsurance:string,AgesAllowed:string,Alcohol:string,Ambience:string,BYOB:string,BYOBCorkage:string,BestNights:string,BikeParking:string,BusinessAcceptsBitcoin:string,BusinessAcceptsCreditCards:string,BusinessParking:string,ByAppointmentOnly:string,Caters:string,CoatCheck:string,Corkage:string,DietaryRestrictions:string,DogsAllowed:string,DriveThru:string,GoodForDancing:string,GoodForKids:string,GoodForMeal:string,HairSpecializesIn:string,HappyHour:string,HasTV:string,Music:string,NoiseLevel:string,Open24Hours:string,OutdoorSeating:string,RestaurantsAttire:string,RestaurantsCounterService:string,RestaurantsDelivery:string,RestaurantsGoodForGroups:string,RestaurantsPriceRange2:string,RestaurantsReservations:string,Restaur

In [27]:
%%time

exploded_df = result_df.withColumn("token", explode(split(regexp_replace(col("filtered_text"), r'[\[\]]', ''), ", ")))

lexicon_emotion_list = list(lexicon_emotion.items())
for item in lexicon_emotion_list:
    if not isinstance(item[0], str) or not isinstance(item[1], list):
        raise ValueError("Invalid data format in lexicon_emotion_list")
lexicon_emotion_rdd = spark.sparkContext.parallelize([Row(token=item[0], emotion=item[1]) for item in lexicon_emotion_list])

schema = StructType([
    StructField("token", StringType(), True),
    StructField("emotion", ArrayType(StringType()), True)
])

lexicon_emotion_df = spark.createDataFrame(lexicon_emotion_rdd, schema)
lexicon_emotion_broadcast = broadcast(lexicon_emotion_df)

filtered_df = exploded_df.join(lexicon_emotion_df, exploded_df["token"] == lexicon_emotion_df["token"], "left")

CPU times: user 81.4 ms, sys: 2.05 ms, total: 83.4 ms
Wall time: 231 ms


In [28]:
filtered_df.show(5)

+--------------------+----+-------------------+-----+--------------------+------------+--------------------+------+--------------------+----------------+--------------------+--------------------+---------------+--------------------+-------+----------+-----------+-----------------+-----------+------------+--------------+-----+--------------------+---------+------+--------------------+
|         business_id|cool|               date|funny|           review_id|review_stars|                text|useful|             user_id|         address|          attributes|          categories|           city|               hours|is_open|  latitude|  longitude|             name|postal_code|review_count|business_stars|state|       filtered_text|    token| token|             emotion|
+--------------------+----+-------------------+-----+--------------------+------------+--------------------+------+--------------------+----------------+--------------------+--------------------+---------------+---------------

In [29]:
%%time

exploded_df = filtered_df.withColumn("emotion", explode("emotion"))
emotion_df = exploded_df.groupBy("review_id").pivot("emotion").count().na.fill(0)
emotion_df = emotion_df.drop("filtered_text")

emotion_df.show(5)

+--------------------+-----+------------+-------+----+---+--------+--------+-------+--------+-----+
|           review_id|anger|anticipation|disgust|fear|joy|negative|positive|sadness|surprise|trust|
+--------------------+-----+------------+-------+----+---+--------+--------+-------+--------+-----+
|--Feh6uaXe_XZy0y4...|    2|           5|      2|   4| 11|       7|      17|      6|       4|   11|
|--ez98beazUDWG2sf...|    2|           8|      2|   1| 11|       8|      25|      4|       5|    8|
|--vWFqzrOCBjCgR5c...|    1|           7|      0|   0|  6|       2|       9|      0|       3|    6|
|-0rf04h-vURoqN0wU...|    0|           1|      0|   0|  1|       1|       2|      1|       0|    1|
|-2JylIDuXCyo4OFz1...|    0|           1|      0|   0|  3|       0|       3|      0|       0|    2|
+--------------------+-----+------------+-------+----+---+--------+--------+-------+--------+-----+
only showing top 5 rows

CPU times: user 454 ms, sys: 122 ms, total: 576 ms
Wall time: 29min 45s


In [34]:
emotion_df.write.csv('hdfs://10.150.0.2:9000/emotion_df', header=True)

How many times did each emotion occur? How many reviews are there?

In [37]:
emotion_totals = emotion_df.select([F.sum(col).alias(col) for col in emotion_df.columns])
total_rows = emotion_df.count()
emotion_totals.show()
print("Total number of reviews:", total_rows)

+---------+-------+------------+-------+-------+--------+--------+--------+-------+--------+--------+
|review_id|  anger|anticipation|disgust|   fear|     joy|negative|positive|sadness|surprise|   trust|
+---------+-------+------------+-------+-------+--------+--------+--------+-------+--------+--------+
|     null|5553927|    18957992|4421506|5844501|22138091|13677985|39393887|6335110| 8632445|23122488|
+---------+-------+------------+-------+-------+--------+--------+--------+-------+--------+--------+

Total number of reviews: 6880895
